In [6]:
import os
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

from langchain.agents import create_agent



In [15]:
# os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

search_tool = TavilySearchResults(max_results=2)

@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

tools = [ search_tool, get_weather_data ]

agent = create_agent( 
    model=llm, 
    tools=tools, 
    system_prompt=( 
                "You are a helpful assistant. " 
                "Use the available tools when necessary. " 
                "When asked to find information and then get weather, " 
                "first find the relevant location and then use the weather tool." 
    ) 
)


response = agent.invoke( { "messages": [ { "role": "user", "content": ( "Find the capital of Macedonia " "and then find its current weather." ) } ] } ) 
print(response["messages"][-1].content)

The capital of Macedonia is Skopje. Currently, the weather in Skopje is clear with a temperature of 10°C and 34% humidity.
